In [ ]:
import os
import json
import torch
import faiss
import logging
import numpy as np
from glob import glob
from tqdm import tqdm
from PIL import Image
from os.path import join
import torch.utils.data as data
from torch_geometric.data import Data
from torch_geometric.data import Batch
import torchvision.transforms as transforms
from torch.utils.data.dataset import Subset
from sklearn.neighbors import NearestNeighbors
from torch.utils.data.dataloader import DataLoader
from torch_geometric.loader import DataLoader as GraphDataLoader


from scipy.spatial.distance import pdist
import math

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def path_to_pil_img(path):
    return Image.open(path).convert("RGB")

def collate_graphs_for_triplets(batch):
    """
    batch: list of samples, where each sample is:
      ( list_of_graphs, triplets_local_indexes, triplets_global_indexes_for_sample )
    We need to:
     - flatten all graphs: flat_graphs = concat(sample_graphs for each sample)
     - create Batch.from_data_list(flat_graphs)
     - adjust triplets_local_indexes by adding per-sample offset
     - stack triplets_global_indexes into tensor (optional)
    Returns: (Batch, triplets_local_indexes_tensor, triplets_global_indexes_tensor)
    """
    from torch_geometric.data import Batch
    flat_graphs = []
    offsets = []
    sizes = []
    for sample in batch:
        graphs_i = sample[0]
        sizes.append(len(graphs_i))
        flat_graphs.extend(graphs_i)

    # Build Batch
    batch_graph = Batch.from_data_list(flat_graphs)

    # Build adjusted local triplets indexes
    local_triplets_list = []
    cum = 0
    for sample in batch:
        graphs_i = sample[0]
        local_idx = sample[1]  # tensor shape (negs_num_per_query, 3) or (N,3)
        if isinstance(local_idx, torch.Tensor):
            li = local_idx.clone()
        else:
            li = torch.tensor(local_idx, dtype=torch.long)
        # add offset to li (offset == cum)
        li = li + cum
        local_triplets_list.append(li)
        cum += len(graphs_i)

    # concat into one long tensor
    triplets_local_indexes = torch.cat(local_triplets_list, dim=0)  # shape (batch_size*negs_num, 3)

    # stack global triplet descriptors (if needed)
    global_triplets = torch.stack([torch.tensor(s[2], dtype=torch.long) for s in batch], dim=0)

    return batch_graph, triplets_local_indexes, global_triplets
    
def collate_graph_index(batch):
    """
    Для eval/train inference: dataset[i] -> (graph, global_index)
    Возвращает: (Batch, torch.tensor(global_indexes))
    """
    graphs = [item[0] for item in batch]
    indexes = [int(item[1]) for item in batch]
    return Batch.from_data_list(graphs), torch.tensor(indexes, dtype=torch.long)

def collate_fn(batch):
    """Creates mini-batch tensors from the list of tuples (images, 
        triplets_local_indexes, triplets_global_indexes).
        triplets_local_indexes are the indexes referring to each triplet within images.
        triplets_global_indexes are the global indexes of each image.
    Args:
        batch: list of tuple (images, triplets_local_indexes, triplets_global_indexes).
            considering each query to have 10 negatives (negs_num_per_query=10):
            - images: torch tensor of shape (12, 3, h, w).
            - triplets_local_indexes: torch tensor of shape (10, 3).
            - triplets_global_indexes: torch tensor of shape (12).
    Returns:
        images: torch tensor of shape (batch_size*12, 3, h, w).
        triplets_local_indexes: torch tensor of shape (batch_size*10, 3).
        triplets_global_indexes: torch tensor of shape (batch_size, 12).
    """
    images                  = torch.cat([e[0] for e in batch])
    triplets_local_indexes  = torch.cat([e[1][None] for e in batch])
    triplets_global_indexes = torch.cat([e[2][None] for e in batch])
    for i, (local_indexes, global_indexes) in enumerate(zip(triplets_local_indexes, triplets_global_indexes)):
        local_indexes += len(global_indexes) * i  # Increment local indexes by offset (len(global_indexes) is 12)
    return images, torch.cat(tuple(triplets_local_indexes)), triplets_global_indexes


class PCADataset(data.Dataset):
    def __init__(self, args, datasets_folder="dataset", dataset_folder="pitts30k/images/train"):
        dataset_folder_full_path = join(datasets_folder, dataset_folder)
        if not os.path.exists(dataset_folder_full_path) :
            raise FileNotFoundError(f"Folder {dataset_folder_full_path} does not exist")
        self.images_paths = sorted(glob(join(dataset_folder_full_path, "**", "*.jpg"), recursive=True))
    def __getitem__(self, index):
        return base_transform(path_to_pil_img(self.images_paths[index]))
    def __len__(self):
        return len(self.images_paths)


class BaseDataset(data.Dataset):
    """Dataset with images from database and queries, used for inference (testing and building cache).
    """
    def __init__(self, args, datasets_folder="datasets", dataset_name="graph", split="train"):
        super().__init__()
        self.args = args
        self.dataset_name = dataset_name
        print(os.path.join(datasets_folder, dataset_name, "norms.json"))
        self.norms = self.load_norms(os.path.join(datasets_folder,dataset_name, "norms.json"))
        print(self.norms)
        self.dataset_folder = join(datasets_folder, dataset_name, split)
        if not os.path.exists(self.dataset_folder): raise FileNotFoundError(f"Folder {self.dataset_folder} does not exist")
        self.resize = args.resize
        #self.resize = args.resize
        #self.test_method = args.test_method
        
        #### Read paths and UTM coordinates for all images.
        database_folder = join(self.dataset_folder, "database")
        queries_folder  = join(self.dataset_folder, "queries")
        if not os.path.exists(database_folder): raise FileNotFoundError(f"Folder {database_folder} does not exist")
        if not os.path.exists(queries_folder) : raise FileNotFoundError(f"Folder {queries_folder} does not exist")
        self.database_paths = sorted(glob(join(database_folder, "**", "*.pt"), recursive=True))
        self.queries_paths  = sorted(glob(join(queries_folder, "**", "*.pt"),  recursive=True))
        #print("database_paths", self.database_paths)
        #print("queries_paths", self.queries_pahts)
    
        # The format must be path/to/file/@utm_easting@utm_northing@...@.jpg
        #self.database_utms = np.array([(path.split("@")[1], path.split("@")[2]) for path in self.database_paths]).astype(np.float)
        print("Robit")
   
        
        self.database_utms = self.get_poses(self.database_paths)
        self.queries_utms  = self.get_poses(self.queries_paths)

        
        # Find soft_positives_per_query, which are within val_positive_dist_threshold (deafult 25 meters)
        knn = NearestNeighbors(n_jobs=-1)
        knn.fit(self.database_utms)
        if split=="train":
            print("split == Train. Compute soft_positives_per_query")
            self.soft_positives_per_query = knn.radius_neighbors(self.queries_utms, 
                                                                radius=3,
                                                                return_distance=False)
        else:
            self.soft_positives_per_query = knn.radius_neighbors(self.queries_utms, 
                                                                args.val_positive_dist_threshold,
                                                                return_distance=False)  
        self.images_paths = list(self.database_paths) + list(self.queries_paths)


        self.graphs = [torch.load(p, weights_only=False) for p in self.images_paths]
        self.graphs = [self._build_data(d, i) for i, d in enumerate(self.graphs)]
        
        self.database_num = len(self.database_paths)
        self.queries_num  = len(self.queries_paths)
    
    def __getitem__(self, idx):
        # Получаем словарь с данными (предполагается, что self.graphs — список предзагруженных словарей)
        return self.graphs[idx], idx
        
    def _build_data(self, d, idx):
        data_dict = self.graphs[idx]
    
        # ---- Узлы ----
        x_cont = data_dict['x']  # непрерывные признаки узлов, тензор [N, F_cont]
        if torch.is_tensor(x_cont):
            x_np = x_cont.cpu().numpy()
        else:
            x_np = np.array(x_cont, dtype=float)
    
        # Нормализация непрерывных признаков узлов
        if self.norms is not None and 'node_mean' in self.norms:
            # Индексы столбцов, подлежащих нормализации (по умолчанию все)
            cont_idx = self.norms.get('node_cont_indices', None)
            if cont_idx is None:
                cont_idx = list(range(x_np.shape[1]))
            mean = self.norms['node_mean']
            std = self.norms['node_std']
            for j, feat_idx in enumerate(cont_idx):
                if feat_idx < x_np.shape[1] and j < len(mean):
                    x_np[:, feat_idx] = (x_np[:, feat_idx] - mean[j]) / (std[j] if std[j] > 1e-8 else 1.0)
    
        x_cont = torch.tensor(x_np, dtype=torch.float32)
    
        # Классы узлов
        node_class = data_dict['node_class']
        if not torch.is_tensor(node_class):
            node_class = torch.tensor(node_class, dtype=torch.long)
        else:
            node_class = node_class.long()
    
        # ---- Рёбра ----
        edge_index = data_dict.get('edge_index', None)
        if edge_index is None:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor(edge_index, dtype=torch.long) if not torch.is_tensor(edge_index) else edge_index.long()
    
        # Непрерывные признаки рёбер и метка класса (если есть)
        edge_attr = data_dict.get('edge_attr', None)
        edge_attr_cont = None
        edge_label = None
        if edge_attr is not None:
            if torch.is_tensor(edge_attr):
                ea_np = edge_attr.cpu().numpy()
            else:
                ea_np = np.array(edge_attr, dtype=float)
    
            # Если есть отдельное поле edge_label, используем его
            if 'edge_label' in data_dict:
                edge_label = data_dict['edge_label']
                if not torch.is_tensor(edge_label):
                    edge_label = torch.tensor(edge_label, dtype=torch.long)
                else:
                    edge_label = edge_label.long()
                # Непрерывные признаки — все столбцы edge_attr
                edge_attr_cont = torch.tensor(ea_np, dtype=torch.float32)
            else:
                # Пытаемся определить непрерывные признаки по edge_cont_indices
                if self.norms is not None and 'edge_mean' in self.norms:
                    cont_eidx = self.norms.get('edge_cont_indices', None)
                    if cont_eidx is None:
                        # Если нет индексов, считаем все столбцы непрерывными
                        cont_eidx = list(range(ea_np.shape[1]))
                    emean = self.norms['edge_mean']
                    estd = self.norms['edge_std']
                    # Нормализуем только непрерывные столбцы
                    for j, feat_idx in enumerate(cont_eidx):
                        if feat_idx < ea_np.shape[1] and j < len(emean):
                            ea_np[:, feat_idx] = (ea_np[:, feat_idx] - emean[j]) / (estd[j] if estd[j] > 1e-8 else 1.0)
    
                    # Если последний столбец не вошёл в непрерывные — вероятно, это метка класса
                    if max(cont_eidx) < ea_np.shape[1] - 1:
                        edge_label = torch.tensor(ea_np[:, -1], dtype=torch.long)
                        edge_attr_cont = torch.tensor(ea_np[:, cont_eidx], dtype=torch.float32)
                    else:
                        edge_attr_cont = torch.tensor(ea_np, dtype=torch.float32)
                else:
                    # Нет информации о нормализации — все столбцы считаем непрерывными
                    edge_attr_cont = torch.tensor(ea_np, dtype=torch.float32)
    
        # Классы концевых узлов (если есть)
        edge_u_class = data_dict.get('edge_u_class', None)
        if edge_u_class is not None:
            edge_u_class = torch.tensor(edge_u_class, dtype=torch.long) if not torch.is_tensor(edge_u_class) else edge_u_class.long()
        edge_v_class = data_dict.get('edge_v_class', None)
        if edge_v_class is not None:
            edge_v_class = torch.tensor(edge_v_class, dtype=torch.long) if not torch.is_tensor(edge_v_class) else edge_v_class.long()
    
        # Поза (координаты)
        pose = data_dict.get('pose', None)
        if pose is not None:
            pose = torch.tensor(pose, dtype=torch.float) if not torch.is_tensor(pose) else pose.float()
    
        # ---- Создание объекта Data ----
        from torch_geometric.data import Data
        data = Data(
            x_cont=x_cont,
            node_class=node_class,
            edge_index=edge_index,
            edge_attr_cont=edge_attr_cont,
            edge_label=edge_label,
            edge_u_class=edge_u_class,
            edge_v_class=edge_v_class,
            pose=pose
        )
    
        # Служебные поля
        data.orig_idx = torch.tensor(idx, dtype=torch.long)
        # Если у вас есть список всех путей к файлам (например, self.images_paths), можно добавить имя файла:
        # data.filename = os.path.basename(self.images_paths[idx])
    
        return data
        
    
    def _test_query_transform(self, img):
        """Transform query image according to self.test_method."""
        C, H, W = img.shape
        if self.test_method == "single_query":
            # self.test_method=="single_query" is used when queries have varying sizes, and can't be stacked in a batch.
            processed_img = transforms.functional.resize(img, min(self.resize))
        elif self.test_method == "central_crop":
            # Take the biggest central crop of size self.resize. Preserves ratio.
            scale = max(self.resize[0]/H, self.resize[1]/W)
            processed_img = torch.nn.functional.interpolate(img.unsqueeze(0), scale_factor=scale).squeeze(0)
            processed_img = transforms.functional.center_crop(processed_img, self.resize)
            assert processed_img.shape[1:] == torch.Size(self.resize), f"{processed_img.shape[1:]} {self.resize}"
        elif self.test_method == "five_crops" or self.test_method == 'nearest_crop' or self.test_method == 'maj_voting':
            # Get 5 square crops with size==shorter_side (usually 480). Preserves ratio and allows batches.
            shorter_side = min(self.resize)
            processed_img = transforms.functional.resize(img, shorter_side)
            processed_img = torch.stack(transforms.functional.five_crop(processed_img, shorter_side))
            assert processed_img.shape == torch.Size([5, 3, shorter_side, shorter_side]), \
                f"{processed_img.shape} {torch.Size([5, 3, shorter_side, shorter_side])}"
        return processed_img
    
    def __len__(self):
        return len(self.images_paths)
    def __repr__(self):
        return  (f"< {self.__class__.__name__}, {self.dataset_name} - #database: {self.database_num}; #queries: {self.queries_num} >")

    def load_norms(self, norms_path):
        if not os.path.isfile(norms_path):
            print(f"[norms] warning: {norms_path} not found -> skipping normalization")
            return None
        with open(norms_path, 'r', encoding='utf-8') as fh:
            norms = json.load(fh)
        if 'node_mean' in norms:
            norms['node_mean'] = np.array(norms['node_mean'], dtype=float)
            norms['node_std']  = np.array(norms['node_std'], dtype=float)
        if 'edge_mean' in norms:
            norms['edge_mean'] = np.array(norms['edge_mean'], dtype=float)
            norms['edge_std']  = np.array(norms['edge_std'], dtype=float)
        if 'node_cont_indices' in norms:
            norms['node_cont_indices'] = list(norms['node_cont_indices'])
        if 'edge_cont_indices' in norms:
            norms['edge_cont_indices'] = list(norms['edge_cont_indices'])
        return norms

    def get_positives(self):
        return self.soft_positives_per_query
    def get_database_paths(self):
        return self.database_paths
    def get_poses(self, graphs_paths):
        poses = []
        for p in graphs_paths:
            d = torch.load(p, weights_only=False)
            pose = d.get('pose')[:3] # x, y, z
            if isinstance(pose, torch.Tensor):
                pose = pose.cpu().numpy()
            poses.append(np.array(pose, dtype=float))
        return poses


class TripletsDataset(BaseDataset):
    """Dataset used for training, it is used to compute the triplets 
    with TripletsDataset.compute_triplets() with various mining methods.
    If is_inference == True, uses methods of the parent class BaseDataset,
    this is used for example when computing the cache, because we compute features
    of each image, not triplets.
    """
    def __init__(self, args, datasets_folder="datasets", dataset_name="graph", split="train", negs_num_per_query=10):
        super().__init__(args, datasets_folder, dataset_name, split)
        self.mining = args.mining
        self.debug = True   # ← включить отладку
        self.neg_samples_num = args.neg_samples_num  # Number of negatives to randomly sample
        self.negs_num_per_query = negs_num_per_query  # Number of negatives per query in each batch
        if self.mining == "full":  # "Full database mining" keeps a cache with last used negatives
            self.neg_cache = [np.empty((0,), dtype=np.int32) for _ in range(self.queries_num)]
        self.is_inference = False
        
        identity_transform = transforms.Lambda(lambda x: x)
        self.resized_transform = transforms.Compose([
            transforms.Resize(self.resize) if self.resize is not None else identity_transform,
            base_transform
        ])

        
        self.query_transform = transforms.Compose([
                transforms.ColorJitter(brightness=args.brightness)       if args.brightness          != None else identity_transform,
                transforms.ColorJitter(contrast=args.contrast)           if args.contrast            != None else identity_transform,
                transforms.ColorJitter(saturation=args.saturation)       if args.saturation          != None else identity_transform,
                transforms.ColorJitter(hue=args.hue)                     if args.hue                 != None else identity_transform,
                transforms.RandomPerspective(args.rand_perspective)      if args.rand_perspective    != None else identity_transform,
                transforms.RandomResizedCrop(size=self.resize, scale=(1-args.random_resized_crop, 1))  \
                                                                         if args.random_resized_crop != None else identity_transform,
                transforms.RandomRotation(degrees=args.random_rotation)  if args.random_rotation     != None else identity_transform,
                self.resized_transform,
        ])
    
        
        # Find hard_positives_per_query, which are within train_positives_dist_threshold (10 meters)
        knn = NearestNeighbors(n_jobs=-1)
        knn.fit(self.database_utms)
        self.hard_positives_per_query = list(knn.radius_neighbors(self.queries_utms,
                                             radius=1,  # 10 meters
                                             return_distance=False))
        
        #### Some queries might have no positive, we should remove those queries.
        queries_without_any_hard_positive = np.where(np.array([len(p) for p in self.hard_positives_per_query], dtype=object) == 0)[0]
        if len(queries_without_any_hard_positive) != 0:
            logging.info(f"There are {len(queries_without_any_hard_positive)} queries without any positives. They won't be considered.")
            # Создаём маску для оставшихся элементов
            keep_mask = np.ones(len(self.hard_positives_per_query), dtype=bool)
            keep_mask[queries_without_any_hard_positive] = False
            # Применяем маску к спискам
            self.hard_positives_per_query = [self.hard_positives_per_query[i] for i in range(len(self.hard_positives_per_query)) if keep_mask[i]]
            self.queries_paths = [self.queries_paths[i] for i in range(len(self.queries_paths)) if keep_mask[i]]
            if hasattr(self, 'queries_graphs') and self.queries_graphs is not None:
                self.queries_graphs = [self.queries_graphs[i] for i in range(len(self.queries_graphs)) if keep_mask[i]]
        # Remove queries without positives
        #self.hard_positives_per_query = np.delete(self.hard_positives_per_query, queries_without_any_hard_positive)
        #self.queries_paths            = np.delete(self.queries_paths,            queries_without_any_hard_positive)
        
        # Recompute images_paths and queries_num because some queries might have been removed
        self.images_paths = list(self.database_paths) + list(self.queries_paths)
        self.queries_num = len(self.queries_paths)
        
        # msls_weighted refers to the mining presented in MSLS paper's supplementary.
        # Basically, images from uncommon domains are sampled more often. Works only with MSLS dataset.
        if self.mining == "msls_weighted":
            notes = [p.split("@")[-2] for p in self.queries_paths]
            try:
                night_indexes = np.where(np.array([n.split("_")[0] == "night" for n in notes]))[0]
                sideways_indexes = np.where(np.array([n.split("_")[1] == "sideways" for n in notes]))[0]
            except IndexError:
                raise RuntimeError("You're using msls_weighted mining but this dataset " +
                                   "does not have night/sideways information. Are you using Mapillary SLS?")
            self.weights = np.ones(self.queries_num)
            assert len(night_indexes) != 0 and len(sideways_indexes) != 0, \
                "There should be night and sideways images for msls_weighted mining, but there are none. Are you using Mapillary SLS?"
            self.weights[night_indexes] += self.queries_num / len(night_indexes)
            self.weights[sideways_indexes] += self.queries_num / len(sideways_indexes)
            self.weights /= self.weights.sum()
            logging.info(f"#sideways_indexes [{len(sideways_indexes)}/{self.queries_num}]; " +
                         "#night_indexes; [{len(night_indexes)}/{self.queries_num}]")

    def __getitem__(self, index):
        if self.is_inference:
            # Инференс: возвращаем граф и его индекс (как в BaseDataset)
            return super().__getitem__(index)

        # Тренировка: извлекаем индексы триплета
        q_idx, p_idx, neg_idxs = torch.split(self.triplets_global_indexes[index], (1, 1, self.negs_num_per_query))
        q_idx = q_idx.item()
        p_idx = p_idx.item()
        neg_idxs = neg_idxs.tolist()

        # Получаем графы из предзагруженного списка (self.graphs должен содержать все графы,
        # причём сначала database, затем queries – как в self.images_paths)
        # Индексы database лежат в диапазоне [0, self.database_num-1],
        # индексы запросов – в диапазоне [self.database_num, self.database_num+self.queries_num-1]
        query = self.graphs[self.database_num + q_idx]      # запрос
        positive = self.graphs[p_idx]                         # позитив из базы
        negatives = [self.graphs[i] for i in neg_idxs]        # негативы из базы

        # Формируем локальные индексы для триплетов (аналогично image-версии)
        triplets_local_indexes = torch.empty((0, 3), dtype=torch.int)
        for neg_num in range(len(neg_idxs)):
            triplets_local_indexes = torch.cat((triplets_local_indexes,
                                                torch.tensor([0, 1, 2 + neg_num]).reshape(1, 3)))
        # Возвращаем список графов, локальные и глобальные индексы
        return [query, positive, *negatives], triplets_local_indexes, self.triplets_global_indexes[index]

    
    def __len__(self):
        if self.is_inference:
            # At inference time return the number of images. This is used for caching or computing NetVLAD's clusters
            return super().__len__()
        else:
            return len(self.triplets_global_indexes)
    
    def compute_triplets(self, args, model):
        self.is_inference = True
        if self.mining == "full":
            self.compute_triplets_full(args, model)
        elif self.mining == "partial" or self.mining == "msls_weighted":
            # соберём те же database_indexes, sampled_queries_indexes как в коде
            # diagn = self.debug_analyze_missing(database_indexes + list(sampled_queries_indexes + self.database_num), args, model)
            self.compute_triplets_partial(args, model)
        elif self.mining == "random":
            self.compute_triplets_random(args, model)

  
    @staticmethod
    def compute_cache(args, model, subset_ds, cache_shape):
        """
        Улучшенная версия: логирует какие глобальные индексы были вычислены.
        Возвращает (cache, computed_set).
        subset_ds --- Subset(self, indices_list) или любой Dataset, возвращающий (graph, global_idx) в __getitem__
        """
        subset_dl = DataLoader(
            dataset=subset_ds,
            num_workers=args.num_workers,
            batch_size=args.infer_batch_size,
            shuffle=False,
            pin_memory=(args.device == "cuda"),
            collate_fn=collate_graph_index
        )
        model = model.eval()
        cache = RAMEfficient2DMatrix(cache_shape, dtype=np.float32)
    
        computed_indices = []
    
        with torch.no_grad():
            for batch, indexes in tqdm(subset_dl, ncols=100):
                # indexes — tensor глобальных индексов (как вы делали)
                batch = batch.to(args.device)
                out = model(batch)  # ожидается: [batch_size, feat_dim] или tuple
                # Если model возвращает кортеж, берём первый/global
                if isinstance(out, (tuple, list)):
                    global_features = out[1] if len(out) > 1 else out[0]
                else:
                    global_features = out
                gf = global_features.detach().cpu().numpy()
                idxs = indexes.numpy().astype(int)
                # Запись в кэш

                # в compute_cache, сразу перед cache[idxs] = gf
                if args.debug:
                    print("[CACHE_DEBUG] batch idxs:", idxs, " gf.shape:", getattr(gf, "shape", None))
                    assert gf.shape[0] == len(idxs), f"batch size mismatch: gf rows {gf.shape[0]} vs idxs {len(idxs)}"
                    try:
                        cache[idxs] = gf
                    except Exception as e:
                        # дампим информацию и пробрасываем
                        print("[CACHE_DEBUG] Failed to write batch for idxs:", idxs, "exception:", repr(e))
                        # можно сохранить gf и idxs на диск для дальнейшего анализа
                        raise
                
                cache[idxs] = gf
                computed_indices.extend(idxs.tolist())
    
                if args.debug:
                    print(f"[CACHE] batch wrote indexes: {idxs[:10]}{'...' if len(idxs)>10 else ''}")
    
        computed_set = set(computed_indices)
        if args.debug:
            print(f"[CACHE] total computed rows in this run: {len(computed_set)}")
        return cache, computed_set

    def debug_analyze_missing(self, requested_indices, args, model):
        """
        requested_indices: iterable глобальных индексов, которые вы хотели вычислить.
        Запускает compute_cache над этими индексами (Subet) и показывает:
          - какие индексы были вычислены
          - какие не вычислены (missing)
          - проверки: дубликаты, индексы вне диапазона
        """
        # Нормализуем список индексов
        req = np.asarray(requested_indices, dtype=int).tolist()
        print("\n[DEBUG_ANALYZE] requested count:", len(req))
        uniq = np.unique(req)
        print("[DEBUG_ANALYZE] unique requested:", len(uniq))
        if len(uniq) != len(req):
            print("[DEBUG_ANALYZE] WARNING: there are duplicates among requested indices")
    
        # Проверки границ
        bad_low = [i for i in req if i < 0]
        bad_high = [i for i in req if i >= len(self)]
        if bad_low or bad_high:
            print("[DEBUG_ANALYZE] ERROR: some requested indices out of dataset range")
            print("  <0 examples:", bad_low[:10])
            print("  >=len(dataset) examples:", bad_high[:10])
            return
    
        # Создаём Subset и вызываем compute_cache (он вернёт set вычисленных индексов)
        subset_ds = Subset(self, req)
        cache, computed_set = self.compute_cache(args, model, subset_ds, cache_shape=(len(self), args.features_dim))
    
        requested_set = set(req)
        missing = sorted(list(requested_set - computed_set))
        computed_only = sorted(list(computed_set - requested_set))  # маловероятно, но проверим
    
        print(f"[DEBUG_ANALYZE] requested {len(requested_set)}, computed {len(computed_set)}, missing {len(missing)}")
        if len(missing) > 0:
            print("  Examples of missing (first 50):", missing[:50])
        if len(computed_only) > 0:
            print("  Computed but not requested (examples):", computed_only[:20])
    
        # Проверим формы записанных строк (если хотим)
        shapes = {}
        for i in list(computed_set)[:500]:  # не проверяем всё, только примеры
            v = cache.matrix[i]
            if v is not None:
                shapes.setdefault(v.shape, 0)
                shapes[v.shape] += 1
        print("[DEBUG_ANALYZE] observed shapes in computed subset (sample):", shapes)
    
        # Возвращаем для дальнейшей автоматизированной обработки
        return {
            "requested_count": len(req),
            "unique_requested": len(uniq),
            "computed_count": len(computed_set),
            "missing": missing,
            "computed_only": computed_only,
            "shapes": shapes,
            "cache_obj": cache
        }

    def get_query_features(self, query_index, cache):
        # Индекс запроса в общем списке: database_num + query_index
        full_idx = self.database_num + query_index
        query_features = cache[full_idx]
        if query_features is None:
            raise RuntimeError(f"Features for query index {query_index} not found in cache")
        return query_features
    
    def get_best_positive_index(self, args, query_index, cache, query_features):
        positives_features = cache[self.hard_positives_per_query[query_index]]
        faiss_index = faiss.IndexFlatL2(args.features_dim)
        faiss_index.add(positives_features)
        # Search the best positive (within 10 meters AND nearest in features space)
        _, best_positive_num = faiss_index.search(query_features.reshape(1, -1), 1)
        best_positive_index = self.hard_positives_per_query[query_index][best_positive_num[0]].item()
        return best_positive_index
    
    def get_hardest_negatives_indexes(self, args, cache, query_features, neg_samples):
        if args.debug:
            print("\n[DEBUG] get_hardest_negatives_indexes")
            print("Requested neg_samples:", len(neg_samples))
        
            missing = [int(i) for i in neg_samples if cache.matrix[int(i)] is None]
            print("Missing in cache:", len(missing))
            if len(missing) > 0:
                print("First 20 missing:", missing[:20])
        
            available = [int(i) for i in neg_samples if cache.matrix[int(i)] is not None]
            print("Available negatives:", len(available))
    
            # Проверим форму первой доступной строки
            if len(available) > 0:
                print("Example feature shape:", cache.matrix[available[0]].shape)
        neg_features = cache[neg_samples]
        faiss_index = faiss.IndexFlatL2(args.features_dim)
        faiss_index.add(neg_features)
        # Search the 10 nearest negatives (further than 25 meters and nearest in features space)
        _, neg_nums = faiss_index.search(query_features.reshape(1, -1), self.negs_num_per_query)
        neg_nums = neg_nums.reshape(-1)
        neg_indexes = neg_samples[neg_nums].astype(np.int32)
        return neg_indexes
    
    def compute_triplets_random(self, args, model):
        self.triplets_global_indexes = []
        # Take 1000 random queries
        sampled_queries_indexes = np.random.choice(self.queries_num, args.cache_refresh_rate, replace=False)
        # Take all the positives
        positives_indexes = [self.hard_positives_per_query[i] for i in sampled_queries_indexes]
        positives_indexes = [p for pos in positives_indexes for p in pos]  # Flatten list of lists to a list
        positives_indexes = list(np.unique(positives_indexes))
        
        # Compute the cache only for queries and their positives, in order to find the best positive
        subset_ds = Subset(self, positives_indexes + list(sampled_queries_indexes + self.database_num))
        cache, _ = self.compute_cache(args, model, subset_ds, (len(self), args.features_dim))
        
        # This loop's iterations could be done individually in the __getitem__(). This way is slower but clearer (and yields same results)
        for query_index in tqdm(sampled_queries_indexes, ncols=100):
            query_features = self.get_query_features(query_index, cache)
            best_positive_index = self.get_best_positive_index(args, query_index, cache, query_features)
            
            # Choose some random database images, from those remove the soft_positives, and then take the first 10 images as neg_indexes
            soft_positives = self.soft_positives_per_query[query_index]
            neg_indexes = np.random.choice(self.database_num, size=self.negs_num_per_query+len(soft_positives), replace=False)
            neg_indexes = np.setdiff1d(neg_indexes, soft_positives, assume_unique=True)[:self.negs_num_per_query]
            
            self.triplets_global_indexes.append((query_index, best_positive_index, *neg_indexes))
        # self.triplets_global_indexes is a tensor of shape [1000, 12]
        self.triplets_global_indexes = torch.tensor(self.triplets_global_indexes)
    
    def compute_triplets_full(self, args, model):
        self.triplets_global_indexes = []
        # Take 1000 random queries
        sampled_queries_indexes = np.random.choice(self.queries_num, args.cache_refresh_rate, replace=False)
        # Take all database indexes
        database_indexes = list(range(self.database_num))
        #  Compute features for all images and store them in cache
        subset_ds = Subset(self, database_indexes + list(sampled_queries_indexes + self.database_num))
        cache, _ = self.compute_cache(args, model, subset_ds, (len(self), args.features_dim))
        
        # This loop's iterations could be done individually in the __getitem__(). This way is slower but clearer (and yields same results)
        for query_index in tqdm(sampled_queries_indexes, ncols=100):
            query_features = self.get_query_features(query_index, cache)
            best_positive_index = self.get_best_positive_index(args, query_index, cache, query_features)
            # Choose 1000 random database images (neg_indexes)
            neg_indexes = np.random.choice(self.database_num, self.neg_samples_num, replace=False)
            # Remove the eventual soft_positives from neg_indexes
            soft_positives = self.soft_positives_per_query[query_index]
            neg_indexes = np.setdiff1d(neg_indexes, soft_positives, assume_unique=True)
            # Concatenate neg_indexes with the previous top 10 negatives (neg_cache)
            neg_indexes = np.unique(np.concatenate([self.neg_cache[query_index], neg_indexes]))
            # Search the hardest negatives
            neg_indexes = self.get_hardest_negatives_indexes(args, cache, query_features, neg_indexes)
            # Update nearest negatives in neg_cache
            self.neg_cache[query_index] = neg_indexes
            self.triplets_global_indexes.append((query_index, best_positive_index, *neg_indexes))
        # self.triplets_global_indexes is a tensor of shape [1000, 12]
        self.triplets_global_indexes = torch.tensor(self.triplets_global_indexes)
    
    def compute_triplets_partial(self, args, model):
        self.triplets_global_indexes = []
        # Take 1000 random queries
        if self.mining == "partial":
            sampled_queries_indexes = np.random.choice(self.queries_num, args.cache_refresh_rate, replace=False)
        elif self.mining == "msls_weighted":  # Pick night and sideways queries with higher probability
            sampled_queries_indexes = np.random.choice(self.queries_num, args.cache_refresh_rate, replace=False, p=self.weights)
        
        # Sample 1000 random database images for the negatives
        sampled_database_indexes = np.random.choice(self.database_num, self.neg_samples_num, replace=False)
        # Take all the positives
        positives_indexes = [self.hard_positives_per_query[i] for i in sampled_queries_indexes]
        positives_indexes = [p for pos in positives_indexes for p in pos]
        # Merge them into database_indexes and remove duplicates
        database_indexes = list(sampled_database_indexes) + positives_indexes
        database_indexes = list(np.unique(database_indexes))

        # debug-проверки перед compute_cache
        if args.debug:
            diagn = self.debug_analyze_missing(database_indexes + list(sampled_queries_indexes + self.database_num), args, model)
            db_idx_list = list(database_indexes)
            q_global = (sampled_queries_indexes + self.database_num).tolist() if hasattr(sampled_queries_indexes, 'tolist') else list(sampled_queries_indexes + self.database_num)
            combined = db_idx_list + list(q_global)
            combined = [int(x) for x in combined]
            print(f"[DEBUG_PRECACHE] database_indexes len={len(db_idx_list)}, sampled_queries len={len(q_global)}, combined total={len(combined)}")
            print(f"[DEBUG_PRECACHE] combined unique: {len(set(combined))}")
            # покажем min/max
            print(f"[DEBUG_PRECACHE] min index: {min(combined)}, max index: {max(combined)} (dataset len={len(self)})")
        
        subset_ds = Subset(self, database_indexes + list(sampled_queries_indexes + self.database_num))
        cache, _ = self.compute_cache(args, model, subset_ds, cache_shape=(len(self), args.features_dim))
        
        # This loop's iterations could be done individually in the __getitem__(). This way is slower but clearer (and yields same results)
        for query_index in tqdm(sampled_queries_indexes, ncols=100):
            # Запустить перед best_positive_index = ...
            if args.debug:
                pos_idxs = list(self.hard_positives_per_query[query_index])
                shapes = {}
                bad = []
                for i in pos_idxs:
                    v = cache.matrix[int(i)]
                    if v is None:
                        shapes.setdefault("None", 0)
                        shapes["None"] += 1
                        bad.append((i, "None"))
                    else:
                        shapes.setdefault(v.shape, 0)
                        shapes[v.shape] += 1
                        if v.ndim != 1 or v.shape[0] != args.features_dim:
                            bad.append((i, v.shape))
                print(f"[DIAG] query {query_index}: positives count {len(pos_idxs)}; shapes summary (sample): {dict(list(shapes.items())[:10])}")
                if len(bad) > 0:
                    print("[DIAG] problematic positive examples (index, shape):", bad[:50])
            query_features = self.get_query_features(query_index, cache)
            best_positive_index = self.get_best_positive_index(args, query_index, cache, query_features)
            
            # Choose the hardest negatives within sampled_database_indexes, ensuring that there are no positives
            soft_positives = self.soft_positives_per_query[query_index]
            neg_indexes = np.setdiff1d(sampled_database_indexes, soft_positives, assume_unique=True)
            
            # Take all database images that are negatives and are within the sampled database images (aka database_indexes)
            neg_indexes = self.get_hardest_negatives_indexes(args, cache, query_features, neg_indexes)
            self.triplets_global_indexes.append((query_index, best_positive_index, *neg_indexes))
        # self.triplets_global_indexes is a tensor of shape [1000, 12]
        self.triplets_global_indexes = torch.tensor(self.triplets_global_indexes)


class RAMEfficient2DMatrix:
    """This class behaves similarly to a numpy.ndarray initialized
    with np.zeros(), but is implemented to save RAM when the rows
    within the 2D array are sparse. In this case it's needed because
    we don't always compute features for each image, just for few of
    them"""
    def __init__(self, shape, dtype=np.float32):
        self.shape = shape
        self.dtype = dtype
        self.matrix = [None] * shape[0]
    def __setitem__(self, indexes, vals):
        if vals.ndim != 2 or vals.shape[1] != self.shape[1]:
            raise ValueError(f"vals shape {vals.shape} incompatible with features_dim {self.shape[1]}")
        for i, val in zip(indexes, vals):
            self.matrix[int(i)] = val.astype(self.dtype, copy=False)

    def __getitem__(self, index):
        if hasattr(index, "__len__"):
            rows = [self.matrix[int(i)] for i in index]
            missing = [int(i) for i, r in zip(index, rows) if r is None]
            if missing:
                # не молчим — явно возвращаем информацию
                raise KeyError(f"RAMEfficient2DMatrix: missing rows for indices: {missing}")
            return np.vstack(rows)
        else:
            return self.matrix[int(index)]

class RAMEfficient4DMatrix:
    """This class behaves similarly to a numpy.ndarray initialized
    with np.zeros(), but is implemented to save RAM when the rows
    within the 3D array are sparse. In this case it's needed because
    we don't always compute features for each image, just for few of
    them"""
    def __init__(self, shape, dtype=np.float32):
        self.shape = shape
        self.dtype = dtype
        self.matrix = [None] * shape[0]
    def __setitem__(self, indexes, vals):
        assert vals.shape[1] == self.shape[1], f"{vals.shape[1]} {self.shape[1]}"
        assert vals.shape[2] == self.shape[2], f"{vals.shape[2]} {self.shape[2]}"
        assert vals.shape[3] == self.shape[3], f"{vals.shape[3]} {self.shape[3]}"
        for i, val in zip(indexes, vals):
            self.matrix[i] = val.astype(self.dtype, copy=False)
    def __getitem__(self, index):
        if hasattr(index, "__len__"):
            return np.array([self.matrix[i] for i in index])
        else:
            return self.matrix[index]

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    # Основные параметры для майнинга триплетов
    mining="partial",                       # метод майнинга: "full", "partial", "msls_weighted", "random"
    neg_samples_num=1000,                    # количество случайных негативов для майнинга
    train_positives_dist_threshold=3,       # радиус (в метрах) для определения позитивов при тренировке
    val_positive_dist_threshold=3,           # радиус для валидации (используется в BaseDataset)
    cache_refresh_rate=1000,                  # сколько запросов обрабатывать за один раз при обновлении триплетов
    features_dim=128,                         # размерность выходных эмбеддингов модели
    num_workers=4,                             # число воркеров для DataLoader
    infer_batch_size=32,                       # размер батча при инференсе (вычислении кэша)
    device="cuda",                              # или "cpu"
    debug=False,
    dataset_name="graph_new",
    datasets_folder="/home/amelekhin96/pinkin_ek/data/",
    negs_num_per_query=2,
    seed=42,
    recall_values=[1, 5, 10, 20],
    rerank_num=100,
    
    # Параметры для BaseDataset (не используются для графов, но нужны для совместимости)
    resize=(480, 480),                          # заглушка
    test_method="hard_resize",                   # заглушка

    # Train parametrs:
    optim="adam",
    criterion="triplet",
    train_batch_size=250,
    lr=0.00001,
    margin=0.1,
    epochs_num=100,
    queries_per_epoch=5000,
    
    # Параметры аугментаций (не используются, можно оставить None)
    brightness=None,
    contrast=None,
    saturation=None,
    hue=None,
    rand_perspective=None,
    random_resized_crop=None,
    random_rotation=None,
)

# Если планируется использовать "msls_weighted", убедитесь, что пути содержат метки night/sideways.
# В противном случае лучше выбрать другой метод майнинга.

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

class SimpleGraphEncoder(nn.Module):
    def __init__(self, node_feat_dim, num_node_classes, edge_feat_dim=None, num_edge_classes=None,
                 node_embed_dim=16, edge_embed_dim=8, hidden_dim=128, out_dim=512, num_layers=3,
                 use_learnable_empty=True):
        super().__init__()
        self.node_feat_dim = node_feat_dim
        self.num_node_classes = num_node_classes
        self.edge_feat_dim = edge_feat_dim
        self.num_edge_classes = num_edge_classes
        self.out_dim = out_dim
        self.use_learnable_empty = use_learnable_empty

        # embeddings for node classes
        self.node_class_embed = nn.Embedding(num_node_classes, node_embed_dim)

        # projection for node features
        combined_node_dim = node_feat_dim + node_embed_dim
        self.node_proj = nn.Linear(combined_node_dim, hidden_dim)

        # edges (unused in conv but prepare if needed)
        if edge_feat_dim is not None:
            combined_edge_dim = edge_feat_dim
            if num_edge_classes is not None:
                self.edge_class_embed = nn.Embedding(num_edge_classes, edge_embed_dim)
                combined_edge_dim += edge_embed_dim
            self.edge_proj = nn.Linear(combined_edge_dim, hidden_dim)
            self.use_edge_attr = True
        else:
            self.use_edge_attr = False

        # GCN layers
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            # all layers have same hidden_dim for simplicity
            self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # pool and MLP
        self.pool = lambda x, batch: torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        self.fc = nn.Linear(hidden_dim * 2, out_dim)

        # learnable vector for empty graphs (optional)
        if self.use_learnable_empty:
            # initialize small values
            self.empty_graph_vector = nn.Parameter(torch.zeros(out_dim), requires_grad=True)
        else:
            self.register_buffer("empty_graph_vector", torch.zeros(out_dim))
        

    def forward(self, data):
        """
        data: Data or Batch (PyG). Should contain:
          - data.x_cont: [N, node_feat_dim]
          - data.node_class: [N] long
          - data.edge_index: [2, E] (may be empty)
          - data.batch: [N] long (if batched)
        Handles cases where N == 0 (no nodes).
        """
        device = next(self.parameters()).device

        # === handle absent / empty node features ===
        x_cont = getattr(data, "x_cont", None)
        node_class = getattr(data, "node_class", None)
        edge_index = getattr(data, "edge_index", None)

        # If no nodes in the whole batch:
        if x_cont is None or x_cont.numel() == 0 or x_cont.shape[0] == 0:
            print(data, "Empty object")
            # determine batch_size (number of graphs)
            batch_size = getattr(data, "num_graphs", None)
            if batch_size is None:
                # fallback: single graph
                batch_size = 1
            # return repeated empty vector (learnable or zeros)
            out = self.empty_graph_vector.unsqueeze(0).expand(batch_size, -1).to(device)
            return out

        # otherwise, there are some nodes in the batch
        # ensure types and device
        x_cont = x_cont.to(device)
        if node_class is None:
            # fallback to zeros class if missing
            node_class = torch.zeros(x_cont.size(0), dtype=torch.long, device=device)
        else:
            node_class = node_class.to(device)

        # prepare batch vector (for pooling)
        if hasattr(data, "batch"):
            batch = data.batch.to(device)
        else:
            batch = torch.zeros(x_cont.size(0), dtype=torch.long, device=device)

        # node class embeddings (safe index)
        # in case node_class contains values >= num_node_classes, clamp (optional)
        # node_class = node_class.clamp(0, self.num_node_classes - 1)
        node_class_emb = self.node_class_embed(node_class)  # [N, node_embed_dim]

        # cat cont+class emb and project
        x = torch.cat([x_cont, node_class_emb], dim=1)  # [N, node_feat_dim + embed]
        x = self.node_proj(x)
        x = F.relu(x)

        # run convs (edge_index might be empty but GCNConv can handle E==0 if there are nodes)
        if edge_index is None:
            # create empty edge_index on correct device/shape
            edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        else:
            edge_index = edge_index.to(device)

        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)

        # --- после выполнения convs ---
        # prepare batch vector (для pooling)
        # определим batch_size (кол-во графов в батче)
        batch_size = getattr(data, "num_graphs", None)
        if batch_size is None:
            # fallback: если batch пустой (нет узлов) — считать 1, иначе max(batch)+1
            batch_size = 1 if batch.numel() == 0 else int(batch.max().item()) + 1
        
        # Используем size=batch_size, чтобы pooling вернул ровно batch_size строк
        x_mean = global_mean_pool(x, batch, size=batch_size)
        x_max  = global_max_pool(x, batch, size=batch_size)
        x_pool = torch.cat([x_mean, x_max], dim=1)  # shape: [batch_size, hidden_dim*2]
        
        # окончательная проекция
        out = self.fc(x_pool)  # shape: [batch_size, out_dim]
        return out

In [13]:
#dataset = TripletsDataset(args = args, datasets_folder = '/home/amelekhin96/pinkin_ek/data/')
#dataset.is_inference = True
#dataset.compute_triplets(args, model)
#dataset.is_inference = False
#print(dataset[0])

In [14]:

import torch
import random
from glob import glob

import torch.nn.functional as F
import datetime
import numpy as np
import cv2
import math

def get_keypoints(img_size):
    # flaten by x 
    H,W = img_size
    patch_size = 1#14
    N_h = H//patch_size
    N_w = W//patch_size
    keypoints = np.zeros((2, N_h*N_w), dtype=int)
    keypoints[0] = np.tile(np.linspace(patch_size//2, W-patch_size//2, N_w, 
                                       dtype=int), N_h)
    keypoints[1] = np.repeat(np.linspace(patch_size//2, H-patch_size//2, N_h,
                                         dtype=int), N_w)
    return np.transpose(keypoints)

def match_batch_tensor(fm1, fm2, trainflag, grid_size):
    '''
    fm1: (l,D)
    fm2: (N,l,D)
    mask1: (l)
    mask2: (N,l)
    '''
    M = torch.matmul(fm2, fm1.T) # (N,l,l)
    
    max1 = torch.argmax(M, dim=1) #(N,l)
    max2 = torch.argmax(M, dim=2) #(N,l)
    m = max2[torch.arange(M.shape[0]).reshape((-1,1)), max1] #(N, l)
    valid = torch.arange(M.shape[-1]).repeat((M.shape[0],1)).cuda() == m #(N, l) bool
    
    scores = torch.zeros(fm2.shape[0]).cuda()

    kps = get_keypoints(grid_size)
    for i in range(fm2.shape[0]):
        idx1 = torch.nonzero(valid[i,:]).squeeze()
        idx2 = max1[i,:][idx1]
        assert idx1.shape==idx2.shape

        if trainflag:
            if len(idx1.shape)>0:      
                similarity = torch.mean(torch.sum(fm1[idx1] * fm2[i][idx2],dim=1),dim=0)
            else:
                print("No mutual nearest neighbors!")
                similarity = torch.mean(torch.sum(fm1 * fm2[i],dim=1),dim=0)
            return similarity
        
        else:
            if len(idx1.shape)<1:
                scores[i] = 0
            else:
                scores[i] = len(idx1)
    return scores

def local_sim(features_1, features_2, trainflag=False):
    B, H, W, C = features_2.shape
    if trainflag:
        queries = features_1
        preds = features_2
        queries,preds = queries.view(B, H*W, C),preds.view(B, H*W, C)
        similarity = torch.zeros(B).cuda()
        for i in range(B):
            query,pred = queries[i],preds[i].unsqueeze(0)
            similarity[i] = match_batch_tensor(query, pred, trainflag, grid_size=(H, W))
        return similarity
    else:
        query = features_1
        preds = features_2
        query,preds = query.view(H*W, C),preds.view(B, H*W, C)
        scores = match_batch_tensor(query, preds,trainflag, grid_size=(H, W))
        return scores

In [15]:
import faiss
import torch
import logging
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Subset

import math
import datetime
import os
from os.path import join


def test(args, eval_ds, model, test_method="hard_resize", pca=None):
    """Compute features of the given dataset and compute the recalls."""

    assert test_method in ["hard_resize", "single_query", "central_crop", "five_crops",
                            "nearest_crop", "maj_voting"], f"test_method can't be {test_method}"

    #if args.efficient_ram_testing:
    #    return test_efficient_ram_usage(args, eval_ds, model, test_method)    

    model = model.eval()
    with torch.no_grad():
        logging.debug("Extracting database features for evaluation/testing (graphs)")
    
        # database subset (global indices 0 .. database_num-1)
        eval_ds.test_method = "graph"  # just semantic; not used but keeps parity
        database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
    
        database_dataloader = DataLoader(
            dataset=database_subset_ds,
            num_workers=args.num_workers,
            batch_size=args.infer_batch_size,
            collate_fn=collate_graph_index,   # returns (Batch, tensor(indexes))
            pin_memory=(args.device == "cuda"),
            shuffle=False
        )
    
        # allocate feature array for all images (database + queries)
        all_features = np.empty((len(eval_ds), args.features_dim), dtype=np.float32)
    
        # iterate over graph batches
        for batch_graphs, indices in tqdm(database_dataloader, ncols=100):
            # move PyG Batch to device
            batch_graphs = batch_graphs.to(args.device)
    
            # model may return tensor or tuple/list (handle both)
            out = model(batch_graphs)
            if isinstance(out, (tuple, list)):
                # prefer second element if present (some models return (local, global))
                features = out[1] if len(out) > 1 else out[0]
            else:
                features = out
    
            # features: tensor [batch_size, feat_dim]
            features = features.detach().cpu().numpy()
            if pca is not None:
                features = pca.transform(features)
    
            # indices is tensor of global indices (as returned by collate_graph_index)
            idxs = indices.numpy().astype(int)
            all_features[idxs, :] = features
    
        logging.debug("Extracting query features for evaluation/testing (graphs)")
    
        # queries: subset of global indices database_num .. database_num+queries_num-1
        queries_infer_batch_size = 1 if test_method == "single_query" else args.infer_batch_size
        eval_ds.test_method = "graph"  # semantic only
        queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num + eval_ds.queries_num)))
    
        queries_dataloader = DataLoader(
            dataset=queries_subset_ds,
            num_workers=args.num_workers,
            batch_size=queries_infer_batch_size,
            collate_fn=collate_graph_index,
            pin_memory=(args.device == "cuda"),
            shuffle=False
        )
    
        for batch_graphs, indices in tqdm(queries_dataloader, ncols=100):
            batch_graphs = batch_graphs.to(args.device)
    
            out = model(batch_graphs)
            if isinstance(out, (tuple, list)):
                features = out[1] if len(out) > 1 else out[0]
            else:
                features = out
    
            features = features.detach().cpu().numpy()
            if pca is not None:
                features = pca.transform(features)
    
            # indices are global indices already, so we can store directly
            idxs = indices.numpy().astype(int)
            all_features[idxs, :] = features
    
    queries_features = all_features[eval_ds.database_num:]
    database_features = all_features[:eval_ds.database_num]

    faiss_index = faiss.IndexFlatL2(args.features_dim)
    faiss_index.add(database_features)
    del database_features, all_features
    
    logging.debug("Calculating recalls")
    distances, predictions = faiss_index.search(queries_features, args.rerank_num)
    # вставить сразу после получения `distances, predictions = faiss_index.search(...)`
    def spatial_dist(p1, p2):
        p1 = np.asarray(p1).ravel()
        p2 = np.asarray(p2).ravel()
        return float(np.linalg.norm(p1[:2] - p2[:2]))  # XY distance, или use [:3] для 3D
    
    N_debug = min(10, eval_ds.queries_num)
    for q_idx in range(N_debug):
        q_path = eval_ds.queries_paths[q_idx]
        print(f"\n=== QUERY {q_idx}: {os.path.basename(q_path)} ===")
        preds = predictions[q_idx][:10]
        dists = distances[q_idx][:10]
        pos_idxs = eval_ds.soft_positives_per_query[q_idx]  # индексы по базе (0..database_num-1)
        pos_set = set(int(x) for x in pos_idxs)
        # get query pose
        q_pose = eval_ds.get_poses([q_path])[0] if hasattr(eval_ds, 'get_poses') else None
        for rank, (db_idx, feat_dist) in enumerate(zip(preds, dists), 1):
            db_path = eval_ds.database_paths[int(db_idx)]
            in_pos = (int(db_idx) in pos_set)
            # spatial distance between query and this db candidate:
            try:
                db_pose = eval_ds.get_poses([db_path])[0]
                spd = spatial_dist(q_pose, db_pose)
            except Exception:
                spd = None
            print(f" {rank:2d}. {os.path.basename(db_path)} (db_idx={db_idx}) feat_dist={feat_dist:.4f} in_pos={in_pos} spatial={spd}")

    print("Example positives counts (first 10 queries):")
    for i in range(10):
        print(len(eval_ds.soft_positives_per_query[i]))
    verbose_topk = 10
    if verbose_topk > 0:
        print(f"\nTop-{verbose_topk} candidates for each query (showing up to 10 queries):")
        positives_per_query = eval_ds.get_positives()
        num_to_show = min(10, eval_ds.queries_num)
        for q_idx in range(num_to_show):
            q_path = eval_ds.queries_paths[q_idx]
            print(f"Query {q_idx}: {os.path.basename(q_path)}")
            preds = predictions[q_idx][:verbose_topk]
            dists = distances[q_idx][:verbose_topk]
            pos_set = set(positives_per_query[q_idx])
            for rank, (db_idx, dist) in enumerate(zip(preds, dists), 1):
                db_path = eval_ds.database_paths[db_idx]
                correct = "✓" if db_idx in pos_set else "✗"
                print(f"  {rank}: {os.path.basename(db_path)} (dist={dist:.2f}) {correct}")
            print()


    #### For each query, check if the predictions are correct
    positives_per_query = eval_ds.get_positives()
    # args.recall_values by default is [1, 5, 10, 20]
    recalls = np.zeros(len(args.recall_values))
    for query_index, pred in enumerate(predictions):
        for i, n in enumerate(args.recall_values):
            if np.any(np.in1d(pred[:n], positives_per_query[query_index])):
                recalls[i:] += 1
                break
    # Divide by the number of queries*100, so the recalls are in percentages
    recalls = recalls / eval_ds.queries_num * 100
    recalls_str =", ".join([f"R@{val}: {rec:.1f}" for val, rec in zip(args.recall_values, recalls)])
    logging.info(f"First ranking recalls: {recalls_str}")

    return recalls, recalls_str

In [18]:
import math
import torch
import logging
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import multiprocessing
from os.path import join
from datetime import datetime
import torchvision.transforms as transforms
from torch.utils.data.dataloader import DataLoader
torch.backends.cudnn.benchmark= True  # Provides a speedup

import warnings
warnings.filterwarnings('ignore')
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

#### Creation of Datasets
logging.debug(f"Loading dataset {args.dataset_name} from folder {args.datasets_folder}")

triplets_ds = TripletsDataset(args, args.datasets_folder, args.dataset_name, "train", args.negs_num_per_query)
logging.info(f"Train query set: {triplets_ds}")

val_ds = BaseDataset(args, args.datasets_folder, args.dataset_name, "test")
logging.info(f"Val set: {val_ds}")

#### Initialize model
model = model = SimpleGraphEncoder(
    node_feat_dim=13,               # размер x_cont
    num_node_classes=4585,
    edge_feat_dim=9,                 # если есть edge_attr_cont
    num_edge_classes=4585, # если есть edge_label
    out_dim=args.features_dim
).to(args.device)

#model = torch.nn.DataParallel(model)



### Setup Optimizer and Loss
if args.optim == "adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=int(args.queries_per_epoch/args.train_batch_size), gamma=0.5, last_epoch=-1)
elif args.optim == "sgd":
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=0.001)

GlobalTriplet = nn.TripletMarginLoss(margin=args.margin, p=2, reduction="sum")

#### Resume model, optimizer, and other training parameters
#if args.resume:
#    model, _, best_r5, start_epoch_num, not_improved_num = util.resume_train(args, model, strict=False)
#    logging.info(f"Resuming from epoch {start_epoch_num} with best recall@5 {best_r5:.1f}")
#else:
best_r5 = start_epoch_num = not_improved_num = 0

logging.info(f"Output dimension of the model is {args.features_dim}")

#### Training loop
for epoch_num in range(start_epoch_num, args.epochs_num):
    logging.info(f"Start training epoch: {epoch_num:02d}")
    
    epoch_start_time = datetime.now()
    epoch_losses = np.zeros((0,1), dtype=np.float32)
    
    # How many loops should an epoch last (default is 5000/1000=5)
    loops_num = math.ceil(args.queries_per_epoch / args.cache_refresh_rate)
    for loop_num in range(loops_num):
        logging.debug(f"Cache: {loop_num} / {loops_num}")
        
        # Compute triplets to use in the triplet loss
        triplets_ds.is_inference = True
        triplets_ds.compute_triplets(args, model)
        triplets_ds.is_inference = False

        soft = val_ds.soft_positives_per_query
        hard = triplets_ds.hard_positives_per_query  # если TripletsDataset уже инициализирован
        print("soft: mean,median,max:", np.mean([len(x) for x in soft]), np.median([len(x) for x in soft]), np.max([len(x) for x in soft]))
        print("hard: mean,median,max:", np.mean([len(x) for x in hard]), np.median([len(x) for x in hard]), np.max([len(x) for x in hard]))
        print("queries without hard positives:", sum(1 for x in hard if len(x)==0))
        # 1) Проверим, сколько триплетов сгенерировано
        print("triplets_global_indexes exists?", hasattr(triplets_ds, "triplets_global_indexes"))
        if hasattr(triplets_ds, "triplets_global_indexes"):
            tg = triplets_ds.triplets_global_indexes
            print("type:", type(tg))
            try:
                print("len(triplets_global_indexes):", len(tg))
                if len(tg) > 0:
                    # покажем первые 5
                    print("first 5 triplets (raw):", tg[:5])
            except Exception as e:
                print("Can't get len/peek of triplets_global_indexes:", e)
        
        # 2) Проверим __len__ датасета и train_batch_size / drop_last
        print("len(dataset) according to __len__():", len(triplets_ds))
        print("args.train_batch_size:", args.train_batch_size, "drop_last:", True)
        if len(triplets_ds) < args.train_batch_size:
            print("WARNING: dataset length < train_batch_size -> with drop_last=True DataLoader yields 0 batches")
        
        triplets_dl = DataLoader(dataset=triplets_ds, num_workers=args.num_workers,
                                 batch_size=args.train_batch_size,
                                 collate_fn=collate_graphs_for_triplets,
                                 pin_memory=(args.device=="cuda"),
                                 drop_last=True)
        
        model = model.train()
        # images shape: (train_batch_size*4)*3*H*W
        for images, triplets_local_indexes, _ in tqdm(triplets_dl, ncols=100):
            # Flip all triplets or none
            
            # Compute features of all images (images contains queries, positives and negatives)          
            global_features = model(images.to(args.device))
            total_loss = 0
            global_loss = 0

            if args.criterion == "triplet":
                triplets_local_indexes = torch.transpose(
                    triplets_local_indexes.view(args.train_batch_size, args.negs_num_per_query, 3), 1, 0)
                for triplets in triplets_local_indexes:
                    queries_indexes, positives_indexes, negatives_indexes = triplets.T

                    global_loss += GlobalTriplet(global_features[queries_indexes],
                                                      global_features[positives_indexes],
                                                      global_features[negatives_indexes])

            global_loss /= (args.train_batch_size * args.negs_num_per_query)
                           
            total_loss = global_loss

            del global_features

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            batch_loss = total_loss.item()
            epoch_losses = np.append(epoch_losses, batch_loss)
            del total_loss
        
        logging.debug(f"global loss = {global_loss.item():.6f}")
        logging.debug(f"Epoch[{epoch_num:02d}]({loop_num}/{loops_num}): " +
                      f"current batch triplet loss = {batch_loss:.4f}, " +
                      f"average epoch triplet loss = {epoch_losses.mean():.4f}")
    
    logging.info(f"Finished epoch {epoch_num:02d} in {str(datetime.now() - epoch_start_time)[:-7]}, "
                 f"average epoch triplet loss = {epoch_losses.mean():.4f}")

    # Compute recalls on validation set
    recalls, recalls_str = test(args, val_ds, model)
    logging.info(f"Reranking recalls on val set {val_ds}: {recalls_str}")


    is_best = recalls[1] > best_r5
    
    # Save checkpoint, which contains all training parameters
    """
    util.save_checkpoint(args, {"epoch_num": epoch_num, "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(), "recalls": recalls, "best_r5": best_r5,
        "not_improved_num": not_improved_num
    }, is_best, filename="last_model.pth")
    
    # If recall@5 did not improve for "many" epochs, stop training
    
    if is_best:
        logging.info(f"Improved: previous best R@5 = {best_r5:.1f}, current R@5 = {(recalls[1]):.1f}")
        best_r5 = recalls[1]
        not_improved_num = 0
    else:
        not_improved_num += 1
        logging.info(f"Not improved: {not_improved_num} / {args.patience}: best R@5 = {best_r5:.1f}, current R@5 = {(recalls[1]):.1f}")
        if not_improved_num >= args.patience:
            logging.info(f"Performance did not improve for {not_improved_num} epochs. Stop training.")
            break
    """

logging.info(f"Best R@5: {best_r5:.1f}")
logging.info(f"Trained for {epoch_num+1:02d} epochs, in total in {str(datetime.now() - start_time)[:-7]}")

"""
#### Test best model on test set
best_model_state_dict = torch.load(join(args.save_dir, "best_model.pth"))["model_state_dict"]
model.load_state_dict(best_model_state_dict)

recalls, recalls_str = test.test(args, test_ds, model, test_method=args.test_method)
logging.info(f"Recalls on {test_ds}: {recalls_str}")
"""


DEBUG:root:Loading dataset graph_new from folder /home/amelekhin96/pinkin_ek/data/


/home/amelekhin96/pinkin_ek/data/graph_new/norms.json
{'node_mean': array([0.50926369, 0.50616455, 0.2319064 , 0.31577238, 0.49236819,
       2.56375623, 3.50605798, 0.04054737, 0.08738646, 0.08117685,
       0.02583118, 1.        , 3.76291156]), 'node_std': array([1.97477510e-01, 2.11222906e-01, 2.77782188e-01, 2.96257421e-01,
       1.71030453e-01, 1.07097165e+01, 1.04121858e+01, 1.72112714e-01,
       1.00221293e-01, 9.35901253e-02, 2.75471116e-02, 1.00000000e-06,
       4.55112624e+00]), 'edge_mean': array([ 4.79885621e-01, -2.72083770e-03,  9.13175509e-03,  2.06033455e-02,
        1.63978888e-01,  1.35565772e+01,  1.06798566e-01,  0.00000000e+00]), 'edge_std': array([2.43497543e-01, 7.31615546e-01, 6.81650771e-01, 7.37101201e-02,
       3.45021404e-01, 6.12290249e+01, 1.98002912e-01, 1.00000000e-06]), 'edge_cont_indices': [0, 1, 2, 3, 4, 5, 6, 7]}
Robit
split == Train. Compute soft_positives_per_query


INFO:root:There are 1155 queries without any positives. They won't be considered.
INFO:root:Train query set: < TripletsDataset, graph_new - #database: 3259; #queries: 3935 >


/home/amelekhin96/pinkin_ek/data/graph_new/norms.json
{'node_mean': array([0.50926369, 0.50616455, 0.2319064 , 0.31577238, 0.49236819,
       2.56375623, 3.50605798, 0.04054737, 0.08738646, 0.08117685,
       0.02583118, 1.        , 3.76291156]), 'node_std': array([1.97477510e-01, 2.11222906e-01, 2.77782188e-01, 2.96257421e-01,
       1.71030453e-01, 1.07097165e+01, 1.04121858e+01, 1.72112714e-01,
       1.00221293e-01, 9.35901253e-02, 2.75471116e-02, 1.00000000e-06,
       4.55112624e+00]), 'edge_mean': array([ 4.79885621e-01, -2.72083770e-03,  9.13175509e-03,  2.06033455e-02,
        1.63978888e-01,  1.35565772e+01,  1.06798566e-01,  0.00000000e+00]), 'edge_std': array([2.43497543e-01, 7.31615546e-01, 6.81650771e-01, 7.37101201e-02,
       3.45021404e-01, 6.12290249e+01, 1.98002912e-01, 1.00000000e-06]), 'edge_cont_indices': [0, 1, 2, 3, 4, 5, 6, 7]}
Robit


INFO:root:Val set: < BaseDataset, graph_new - #database: 3259; #queries: 1188 >
INFO:root:Output dimension of the model is 128
INFO:root:Start training epoch: 00
DEBUG:root:Cache: 0 / 5
100%|██████████████████████████████████████████████████████████| 1000/1000 [00:01<00:00, 568.96it/s]


...

soft: mean,median,max: 120.41498316498317 111.5 359
hard: mean,median,max: 35.37433290978399 29.0 165
queries without hard positives: 0
triplets_global_indexes exists? True
type: <class 'torch.Tensor'>
len(triplets_global_indexes): 1000
first 5 triplets (raw): tensor([[1316,  136, 3068,  119],
        [2647,  831, 1784, 1884],
        [1482,    5, 1655, 1837],
        [ 513, 1959, 1627, 1288],
        [2643,  832, 3101, 1335]])
len(dataset) according to __len__(): 1000
args.train_batch_size: 250 drop_last: True


100%|█████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.66it/s]
DEBUG:root:global loss = 0.188558
DEBUG:root:Epoch[04](3/5): current batch triplet loss = 0.1886, average epoch triplet loss = 0.1937
DEBUG:root:Cache: 4 / 5
100%|██████████████████████████████████████████████████████████| 1000/1000 [00:01<00:00, 563.00it/s]


soft: mean,median,max: 120.41498316498317 111.5 359
hard: mean,median,max: 35.37433290978399 29.0 165
queries without hard positives: 0
triplets_global_indexes exists? True
type: <class 'torch.Tensor'>
len(triplets_global_indexes): 1000
first 5 triplets (raw): tensor([[1721,  644, 1612, 3118],
        [3073, 1135, 3219, 3218],
        [1236, 2953, 2351,    1],
        [1526, 2550,  536,  537],
        [ 268, 2095, 1134, 3224]])
len(dataset) according to __len__(): 1000
args.train_batch_size: 250 drop_last: True


100%|█████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.27it/s]


KeyboardInterrupt: 

In [9]:
import os
import glob
import torch
import numpy as np
import pandas as pd

# Путь к файлу запроса (у тебя в примере)
query_path = "/home/amelekhin96/pinkin_ek/data/graph_new/test/queries/map8_000100.pt"

# Папка с database для этого split (можно менять)
db_dir = "/home/amelekhin96/pinkin_ek/data/graph_new/test/database"

# Корневая папка со всеми graph .pt (если хочешь искать по всем картам)
all_graphs_root = "/home/amelekhin96/pinkin_ek/data/graph_new"

# Параметры
top_k = 50          # сколько ближайших вывести; None -> все
spatial_threshold = 3.0  # metres; флаг within_3m

def load_pose_from_pt(p):
    try:
        d = torch.load(p, weights_only=False)
        pose = d.get('pose')
        if pose is None:
            return None
        if isinstance(pose, torch.Tensor):
            pose = pose.cpu().numpy()
        pose = np.asarray(pose, dtype=float).ravel()[:3]
        return pose
    except Exception as e:
        print(f"[WARN] failed to load pose from {p}: {e}")
        return None

def collect_and_sort(query_pose, files_list, by_2d=True):
    rows = []
    for idx, p in enumerate(files_list):
        if os.path.abspath(p) == os.path.abspath(query_path):
            # пропускаем сам запрос, если он попал в список
            continue
        pose = load_pose_from_pt(p)
        if pose is None or len(pose) < 2:
            continue
        # spatial distances: XY (2D) и XYZ (3D)
        dist2 = float(np.linalg.norm(np.array(query_pose[:2]) - np.array(pose[:2])))
        dist3 = float(np.linalg.norm(np.array(query_pose[:3]) - np.array(pose[:3])))
        rows.append({
            "path": p,
            "basename": os.path.basename(p),
            "dist_xy": dist2,
            "dist_xyz": dist3,
            "within_3m": dist2 <= spatial_threshold
        })
    # сортируем по 2D расстоянию (XY)
    rows = sorted(rows, key=lambda r: r["dist_xy"])
    return rows

def print_rows(rows, header=None, top_k=None):
    if header:
        print("\n" + header)
    n = len(rows) if top_k is None else min(len(rows), top_k)
    print(f"Total found: {len(rows)}. Showing top {n}:")
    print(f"{'rank':>4}  {'basename':40s} {'dist_xy(m)':>10s} {'dist_xyz(m)':>11s} {'within_3m':>9s}")
    for i in range(n):
        r = rows[i]
        print(f"{i+1:4d}  {r['basename'][:40]:40s} {r['dist_xy']:10.3f} {r['dist_xyz']:11.3f} {str(r['within_3m']):>9s}")
    if n == 0:
        print("(no entries)")

def save_to_csv(rows, out_csv):
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"[INFO] saved csv -> {out_csv}")

if __name__ == "__main__":
    # 1) load query pose
    q_pose = load_pose_from_pt(query_path)
    if q_pose is None:
        raise RuntimeError(f"Cannot read pose from query {query_path}")

    print(f"Query: {os.path.basename(query_path)} pose (x,y,z): {q_pose}")

    # 2) collect DB files
    db_files = sorted(glob.glob(os.path.join(db_dir, "**", "*.pt"), recursive=True))
    print(f"Database files found in {db_dir}: {len(db_files)}")

    db_rows = collect_and_sort(q_pose, db_files, by_2d=True)
    print_rows(db_rows, header="Nearest in test/database", top_k=top_k)

    # 3) collect ALL .pt under graph_new (optional broader search)
    all_files = sorted(glob.glob(os.path.join(all_graphs_root, "**", "*.pt"), recursive=True))
    print(f"\nAll .pt files under {all_graphs_root}: {len(all_files)} (including queries and db)")
    all_rows = collect_and_sort(q_pose, all_files, by_2d=True)
    print_rows(all_rows, header="Nearest across all graph_new .pt", top_k=top_k)

    # 4) show all within threshold in DB
    within = [r for r in db_rows if r["within_3m"]]
    print(f"\nDatabase entries within {spatial_threshold} m: {len(within)}")
    if len(within) > 0:
        print_rows(within, header=f"Entries within {spatial_threshold} m", top_k=None)

    # 5) optionally save full sorted lists to CSV
    out_db_csv = "/tmp/query_nearest_db_map8_000000.csv"
    out_all_csv = "/tmp/query_nearest_all_map8_000000.csv"
    save_to_csv(db_rows, out_db_csv)
    save_to_csv(all_rows, out_all_csv)

Query: map8_000100.pt pose (x,y,z): [ 18.27754021 -16.24324226  -0.2557272 ]
Database files found in /home/amelekhin96/pinkin_ek/data/graph_new/test/database: 3259

Nearest in test/database
Total found: 3259. Showing top 50:
rank  basename                                 dist_xy(m) dist_xyz(m) within_3m
   1  map1_000327.pt                                0.352       0.352      True
   2  map1_000328.pt                                0.362       0.363      True
   3  map1_000326.pt                                0.382       0.383      True
   4  map1_000329.pt                                0.417       0.417      True
   5  map1_000325.pt                                0.443       0.443      True
   6  map1_000330.pt                                0.506       0.507      True
   7  map1_000324.pt                                0.539       0.540      True
   8  map1_000331.pt                                0.610       0.612      True
   9  map1_000323.pt                                0.6

In [ ]:
# распечатать примеры поз, чтобы понять единицы/масштаб
print("test")
sample = [val_ds.get_poses(val_ds.database_paths)[i] for i in range(0, min(20, len(val_ds.database_paths)))]
print("Примеры поз базы (первые 20):")
for i,p in enumerate(sample):
    print(i, p)
# и для запросов
qsample = [val_ds.get_poses(val_ds.queries_paths)[i] for i in range(0, min(20, len(val_ds.queries_paths)))]
print("Примеры поз запросов (первые 20):")
for i,p in enumerate(qsample):
    print(i, p)

test
Примеры поз базы (первые 20):
0 [ 0.0440825  -0.0384556   0.01084682]
1 [ 0.04992013 -0.03148326  0.01379066]
2 [ 0.05359577 -0.02729612  0.01364068]
3 [ 0.05102538 -0.02401013  0.01068156]
4 [ 0.043455   -0.02717403  0.00882984]
5 [ 0.04050496 -0.03007123  0.00801119]
6 [ 0.04219499 -0.02977314  0.0106741 ]
7 [ 0.04655811 -0.03045516  0.01086668]
8 [ 0.04899847 -0.03104429  0.01298481]
9 [ 0.0507304  -0.03149232  0.0112979 ]
10 [ 0.04952207 -0.03192011  0.01131492]
11 [ 0.04856028 -0.03001955  0.01037722]
12 [ 0.0461061  -0.02913996  0.01003407]
13 [ 0.0446254  -0.02860138  0.01002947]
14 [ 0.04520583 -0.02916031  0.01133869]
15 [ 0.04594835 -0.02974161  0.0118301 ]
16 [ 0.04672398 -0.02913833  0.01375701]
17 [ 0.04800377 -0.02968306  0.01311945]
18 [ 0.04845693 -0.03127383  0.01241887]
19 [ 0.04918518 -0.03201165  0.01093831]


In [11]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

db_poses = np.array(val_ds.get_poses(val_ds.database_paths))   # shape [N_db,2]
q_poses  = np.array(val_ds.get_poses(val_ds.queries_paths))    # shape [N_q,2]

nn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(db_poses)
dists, idxs = nn.kneighbors(q_poses, return_distance=True)
dists = dists.ravel()
print("nearest-dist stats (units of pose): mean,median,90p,99p,max:", np.mean(dists), np.median(dists),
      np.percentile(dists,90), np.percentile(dists,99), np.max(dists))

nearest-dist stats (units of pose): mean,median,90p,99p,max: 0.07577239256201408 0.07250022900953745 0.14025296409183627 0.15940318988637925 0.16030927225619324


In [12]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

db = db_poses
q  = q_poses

knn = NearestNeighbors(n_jobs=-1).fit(db)
# soft radius = 3.0  (валидация)
soft = knn.radius_neighbors(q, radius=3.0, return_distance=False)
soft_counts = [len(x) for x in soft]
# hard radius = 1.0 (train)
hard = knn.radius_neighbors(q, radius=1.0, return_distance=False)
hard_counts = [len(x) for x in hard]

print("soft (3m): mean,median,max:", np.mean(soft_counts), np.median(soft_counts), np.max(soft_counts))
print("hard (1m): mean,median,max:", np.mean(hard_counts), np.median(hard_counts), np.max(hard_counts))
print("queries without hard positives:", sum(1 for x in hard_counts if x==0))

soft (3m): mean,median,max: 123.36616161616162 130.0 184
hard (1m): mean,median,max: 41.05555555555556 36.5 98
queries without hard positives: 0


In [13]:
# сколько уникальных поз среди базы/запросов
print("unique db poses:", len(np.unique(db, axis=0)), " total db:", len(db))
print("unique q poses:",  len(np.unique(q, axis=0)),  " total q:", len(q))
# печать частых поз (если coords дискретны)
from collections import Counter
c = Counter(tuple(x) for x in db)
print("топ-10 наиболее частых координат в базе:", c.most_common(10))

unique db poses: 792  total db: 792
unique q poses: 396  total q: 396
топ-10 наиболее частых координат в базе: [((16.817224502563477, -11.721887588500977, -0.45339709520339966), 1), ((16.815109252929688, -11.718854904174805, -0.4551348090171814), 1), ((16.81879997253418, -11.721624374389648, -0.4530544877052307), 1), ((16.82113265991211, -11.72490119934082, -0.452492356300354), 1), ((16.81903076171875, -11.724401473999023, -0.4530790448188782), 1), ((16.81830596923828, -11.722742080688477, -0.45336979627609253), 1), ((16.8179874420166, -11.722063064575195, -0.4531758725643158), 1), ((16.818300247192383, -11.722423553466797, -0.45351290702819824), 1), ((16.818187713623047, -11.721062660217285, -0.453534871339798), 1), ((16.815673828125, -11.716341972351074, -0.45419058203697205), 1)]


In [14]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# загрузка поз (используем только XY)
db_poses_all = np.array(val_ds.get_poses(val_ds.database_paths))   # shape [N_db, >=2]
q_poses_all  = np.array(val_ds.get_poses(val_ds.queries_paths))    # shape [N_q, >=2]

# use only first two columns (x,y). If your third is height and you want include it,
# change to [:,:3] and scale appropriately.
db_xy = db_poses_all[:, :2]
q_xy  = q_poses_all[:, :2]

# 1) nearest-dist stats
nn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(db_xy)
dists, idxs = nn.kneighbors(q_xy, return_distance=True)
dists = dists.ravel()
print("nearest-dist (XY) stats (units): mean, median, 90p,99p, max:",
      np.mean(dists), np.median(dists), np.percentile(dists,90), np.percentile(dists,99), np.max(dists))

# 2) try different radii and count positives per query
r_candidates = np.concatenate((
    np.linspace(0.01, 0.2, 20),    # very small
    np.linspace(0.2, 1.0, 17),     # small-to-medium
    np.linspace(1.0, 5.0, 9)       # larger
))
r_candidates = np.unique(np.round(r_candidates, 4))
knn = NearestNeighbors(n_jobs=-1).fit(db_xy)

print("\nradius  mean_count  median_count  max_count  queries_with_zero")
results = []
for r in r_candidates:
    neigh = knn.radius_neighbors(q_xy, radius=r, return_distance=False)
    counts = np.array([len(x) for x in neigh])
    results.append((r, counts.mean(), np.median(counts), counts.max(), int((counts==0).sum())))
    print(f"{r:6.3f}   {counts.mean():8.2f}   {np.median(counts):8.1f}   {counts.max():4d}   {int((counts==0).sum()):4d}")

# 3) pick radii that give target mean positives (e.g. target_mean = 10)
target_mean = 10
closest = sorted(results, key=lambda x: abs(x[1]-target_mean))[:5]
print("\nRadii closest to target mean (10):")
for r,m,md,mx,zeros in closest:
    print(f"r={r:.4f} -> mean={m:.2f}, median={md:.1f}, max={mx}, zeros={zeros}")

nearest-dist (XY) stats (units): mean, median, 90p,99p, max: 0.07358931192417871 0.07020814554603946 0.13858825020088944 0.15627241134129505 0.1580094166058029

radius  mean_count  median_count  max_count  queries_with_zero
 0.010       0.93        0.0     23    354
 0.020       1.34        0.0     24    318
 0.030       1.80        0.0     25    290
 0.040       2.18        0.0     26    263
 0.050       2.57        0.0     38    240
 0.060       3.05        0.0     40    219
 0.070       4.11        0.5     43    198
 0.080       4.57        2.0     44    179
 0.090       5.00        2.0     44    164
 0.100       5.41        2.0     44    143
 0.110       5.81        2.0     45    126
 0.120       6.27        2.0     45    108
 0.130       6.76        3.0     45     72
 0.140       7.26        3.0     46     36
 0.150       7.79        4.0     47     12
 0.160       8.27        4.0     48      0
 0.170       8.62        4.0     49      0
 0.180       8.99        4.0     49      0
 0